# Assignment #10 (demo). Gradient boosting

<img src="https://habrastorage.org/webt/ia/m9/zk/iam9zkyzqebnf_okxipihkgjwnw.jpeg" />

**<center>[mlcourse.ai](https://mlcourse.ai) – Open Machine Learning Course** </center><br>

Author: [Yury Kashnitsky](https://yorko.github.io). All content is distributed under the [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) license.


Your task is to beat at least 2 benchmarks in this [Kaggle Inclass competition](https://www.kaggle.com/c/flight-delays-spring-2018). Here, you won’t be provided with detailed instructions. We only give you a brief description of how the second benchmark was achieved using Xgboost. Hopefully, at this stage of the course, it's enough for you to take a quick look at the data in order to understand that this is the type of task where gradient boosting will perform well. Most likely it will be Xgboost; however, we have plenty of categorical features here.

![](https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/mlcourse_ai_jupyter_book/_static/img/xgboost_meme.jpg)

In [1]:
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [2]:
# for Jupyter-book, we copy data from GitHub, locally, to save Internet traffic,
# you can specify the data/ folder from the root of your cloned
# https://github.com/Yorko/mlcourse.ai repo, to save Internet traffic
DATA_PATH = "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/"

In [3]:
train = pd.read_csv(DATA_PATH + "flight_delays_train.csv")
test = pd.read_csv(DATA_PATH + "flight_delays_test.csv")

In [4]:
train.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [5]:
test.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,c-7,c-25,c-3,615,YV,MRY,PHX,598
1,c-4,c-17,c-2,739,WN,LAS,HOU,1235
2,c-12,c-2,c-7,651,MQ,GSP,ORD,577
3,c-3,c-25,c-7,1614,WN,BWI,MHT,377
4,c-6,c-6,c-3,1505,UA,ORD,STL,258


Given flight departure time, carrier's code, departure airport, destination location, and flight distance, you have to predict departure delay for more than 15 minutes. As the simplest benchmark, let's take an Xgboost classifier and two features that are easiest to take: DepTime and Distance. Such a model results in 0.68202 on the LB.

In [6]:
X_train = train[["Distance", "DepTime"]].values
y_train = train["dep_delayed_15min"].map({"Y": 1, "N": 0}).values
X_test = test[["Distance", "DepTime"]].values

X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=17
)

We'll train Xgboost with default parameters on a part of the data and estimate holdout ROC AUC.

In [7]:
xgb_model = XGBClassifier(seed=17)

xgb_model.fit(X_train_part, y_train_part)
xgb_valid_pred = xgb_model.predict_proba(X_valid)[:, 1]

roc_auc_score(y_valid, xgb_valid_pred)

0.7002665283115153

Now we do the same with the whole training set, make predictions to the test set and form a submission file. This is how you beat the first benchmark.

In [8]:
xgb_model.fit(X_train, y_train)
xgb_test_pred = xgb_model.predict_proba(X_test)[:, 1]

pd.Series(xgb_test_pred, name="dep_delayed_15min").to_csv(
    "xgb_2feat.csv", index_label="id", header=True
)

The second benchmark in the leaderboard was achieved as follows:

- Features `Distance` and `DepTime` were taken unchanged
- A feature `Flight` was created from features `Origin` and `Dest`
- Features `Month`, `DayofMonth`, `DayOfWeek`, `UniqueCarrier` and `Flight` were transformed with OHE (`LabelBinarizer`)
- Logistic regression and gradient boosting (xgboost) were trained. Xgboost hyperparameters were tuned via cross-validation. First, the hyperparameters responsible for model complexity were optimized, then the number of trees was fixed at 500 and learning rate was tuned.
- Predicted probabilities were made via cross-validation using `cross_val_predict`. A linear mixture of logistic regression and gradient boosting predictions was set in the form $w_1 * p_{logit} + (1 - w_1) * p_{xgb}$, where $p_{logit}$ is a probability of class 1, predicted by logistic regression, and $p_{xgb}$ – the same for xgboost. $w_1$ weight was selected manually.
- A similar combination of predictions was made for test set.

Following the same steps is not mandatory. That’s just a description of how the result was achieved by the author of this assignment. Perhaps you might not want to follow the same steps, and instead, let’s say, add a couple of good features and train a random forest of a thousand trees.

Good luck!

## My solution: CatBoost with categorical features

In [9]:
from catboost import CatBoostClassifier


def add_features(df):
    df = df.copy()
    df["DepTime"] = df["DepTime"].where(df["DepTime"] < 2400, df["DepTime"] - 2400)
    df["hour"] = df["DepTime"] // 100
    df["minute"] = df["DepTime"] % 100
    df["route"] = df["Origin"] + "-" + df["Dest"]
    df["carrier_origin"] = df["UniqueCarrier"] + "-" + df["Origin"]
    df["carrier_hour"] = df["UniqueCarrier"] + "-" + df["hour"].astype(str)
    df["hour_str"] = df["hour"].astype(str)
    return df


train_fe = add_features(train)
test_fe = add_features(test)

cat_features = ["Month", "DayofMonth", "DayOfWeek", "UniqueCarrier", "Origin", "Dest",
                "route", "carrier_origin", "carrier_hour", "hour_str"]
features = cat_features + ["DepTime", "hour", "minute", "Distance"]

X_full = train_fe[features]
y_full = train["dep_delayed_15min"].map({"Y": 1, "N": 0})
X_test_full = test_fe[features]
X_tr, X_val, y_tr, y_val = train_test_split(X_full, y_full, test_size=0.3, random_state=17, stratify=y_full)

In [10]:
%%time
cb = CatBoostClassifier(iterations=1500, learning_rate=0.05, depth=6, eval_metric="AUC",
                        cat_features=cat_features, random_seed=17, verbose=300)
cb.fit(X_tr, y_tr, eval_set=(X_val, y_val))
cb_valid_pred = cb.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, cb_valid_pred)

0:	test: 0.6534723	best: 0.6534723 (0)	total: 126ms	remaining: 3m 9s


300:	test: 0.8036220	best: 0.8036220 (300)	total: 31s	remaining: 2m 3s


600:	test: 0.8170794	best: 0.8170794 (600)	total: 1m 2s	remaining: 1m 34s


900:	test: 0.8219420	best: 0.8219549 (898)	total: 1m 35s	remaining: 1m 3s


1200:	test: 0.8258118	best: 0.8258118 (1200)	total: 2m 8s	remaining: 31.9s


1499:	test: 0.8273940	best: 0.8274097 (1494)	total: 2m 41s	remaining: 0us

bestTest = 0.8274096612
bestIteration = 1494

Shrink model to first 1495 iterations.


CPU times: user 4min 55s, sys: 18 s, total: 5min 13s
Wall time: 2min 42s


0.8274096612240904

In [11]:
pd.Series(cb.get_feature_importance(), index=features).sort_values(ascending=False)

minute            16.874580
carrier_hour      13.269660
route             12.595874
hour_str          10.625353
DepTime            7.446913
carrier_origin     6.680479
Dest               5.568973
Month              4.778386
Origin             4.573148
UniqueCarrier      4.492293
Distance           4.465360
DayofMonth         3.648278
DayOfWeek          3.634546
hour               1.346158
dtype: float64

Retrain on the whole training set with the best number of iterations and write the submission file.

In [12]:
%%time
best_iter = cb.get_best_iteration() + 1
cb_full = CatBoostClassifier(iterations=best_iter, learning_rate=0.05, depth=6,
                             cat_features=cat_features, random_seed=17, verbose=False)
cb_full.fit(X_full, y_full)
cb_test_pred = cb_full.predict_proba(X_test_full)[:, 1]
pd.Series(cb_test_pred, name="dep_delayed_15min").to_csv("catboost_submission.csv", index_label="id", header=True)
print(best_iter)

1495
CPU times: user 6min 7s, sys: 18.6 s, total: 6min 26s
Wall time: 3min 18s
